# 🔍 Financial Crime Intelligence — Day 1

**Fraud Detection Application** — Rule-based risk scoring with interactive Gradio UI.

This notebook implements a complete fraud detection pipeline:
1. Upload a CSV of transactions
2. Apply rule-based fraud scoring
3. View risk-scored results, summary metrics, and a distribution chart

---

## Section 1: Package Installation

Install all required dependencies. This cell is safe to re-run — pip will skip already-installed packages.

In [ ]:
# ============================================================
# Section 1: Package Installation
# Install pandas, numpy, matplotlib, and gradio.
# The --quiet flag keeps output minimal.
# ============================================================

!pip install --quiet pandas numpy matplotlib gradio

print("✅ All packages installed successfully.")

## Section 2: Imports

Import all libraries needed for data processing, visualization, and the interactive UI.

In [ ]:
# ============================================================
# Section 2: Imports
# pandas  — tabular data manipulation
# numpy   — numerical operations
# matplotlib — chart generation
# gradio  — interactive web UI inside the notebook
# ============================================================

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")  # Use non-interactive backend for chart rendering
import matplotlib.pyplot as plt
import gradio as gr
import tempfile
import os

print(f"pandas  : {pd.__version__}")
print(f"numpy   : {np.__version__}")
print(f"matplotlib: {matplotlib.__version__}")
print(f"gradio  : {gr.__version__}")
print("\n✅ All imports loaded.")

## Section 3: Fraud Detection Logic

Define the rule-based scoring engine. Each transaction is scored on:

| Rule | Condition | Points |
|------|-----------|--------|
| 1 | `amount > 500,000` | +50 |
| 2 | `amount > 1,000,000` | +20 (additional) |
| 3 | `country` in high-risk list | +30 |

**Risk Levels:** LOW (0–29) · MEDIUM (30–69) · HIGH (70–100)

In [ ]:
# ============================================================
# Section 3: Fraud Detection Logic
# Contains the core scoring function and risk classification.
# ============================================================

# High-risk countries list (used in Rule 3)
HIGH_RISK_COUNTRIES = ["Russia", "Nigeria", "North Korea"]


def compute_risk_score(row: pd.Series) -> int:
    """
    Compute a fraud risk score for a single transaction.

    Scoring Rules:
        Rule 1: amount >  500,000   → +50 points
        Rule 2: amount > 1,000,000  → +20 additional points
        Rule 3: country in high-risk list → +30 points

    The maximum possible score is capped at 100.

    Parameters
    ----------
    row : pd.Series
        A row from the transactions DataFrame.

    Returns
    -------
    int
        Risk score between 0 and 100.
    """
    score = 0

    # Rule 1: Large transaction threshold
    if row["amount"] > 500_000:
        score += 50

    # Rule 2: Very large transaction threshold (additional points)
    if row["amount"] > 1_000_000:
        score += 20

    # Rule 3: High-risk country check
    if row["country"] in HIGH_RISK_COUNTRIES:
        score += 30

    # Cap the score at 100
    return min(score, 100)


def classify_risk_level(score: int) -> str:
    """
    Map a numeric risk score to a human-readable risk level.

    Thresholds:
        0  – 29  → LOW
        30 – 69  → MEDIUM
        70 – 100 → HIGH

    Parameters
    ----------
    score : int
        Risk score (0–100).

    Returns
    -------
    str
        One of 'LOW', 'MEDIUM', or 'HIGH'.
    """
    if score >= 70:
        return "HIGH"
    elif score >= 30:
        return "MEDIUM"
    else:
        return "LOW"


print("✅ Fraud detection logic defined.")

## Section 4: Analysis Function

The main analysis pipeline:
1. Parse the uploaded CSV
2. Validate required columns
3. Score every transaction
4. Compute summary metrics
5. Generate the risk distribution chart

In [ ]:
# ============================================================
# Section 4: Analysis Function
# Orchestrates CSV parsing, scoring, metrics, and charting.
# ============================================================

# Required columns that must be present in the uploaded CSV
REQUIRED_COLUMNS = ["transaction_id", "account", "amount", "country", "timestamp"]


def generate_risk_chart(df: pd.DataFrame) -> str:
    """
    Create a styled matplotlib bar chart showing the distribution
    of risk levels (LOW / MEDIUM / HIGH).

    Parameters
    ----------
    df : pd.DataFrame
        Scored transactions DataFrame with a 'risk_level' column.

    Returns
    -------
    str
        File path to the saved chart image.
    """
    # Count transactions per risk level, ensuring consistent ordering
    risk_order = ["LOW", "MEDIUM", "HIGH"]
    counts = df["risk_level"].value_counts().reindex(risk_order, fill_value=0)

    # Color palette: green for low, amber for medium, red for high
    colors = ["#2ecc71", "#f39c12", "#e74c3c"]

    # Build the chart with a dark background for a professional look
    fig, ax = plt.subplots(figsize=(8, 5), facecolor="#1a1a2e")
    ax.set_facecolor("#1a1a2e")

    bars = ax.bar(counts.index, counts.values, color=colors,
                  edgecolor="white", linewidth=0.8, width=0.6)

    # Add value labels on top of each bar
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2.0, height + 0.3,
                f"{int(height)}", ha="center", va="bottom",
                fontweight="bold", fontsize=14, color="white")

    # Styling
    ax.set_title("Risk Level Distribution", fontsize=18,
                 fontweight="bold", color="white", pad=15)
    ax.set_xlabel("Risk Level", fontsize=13, color="white", labelpad=10)
    ax.set_ylabel("Number of Transactions", fontsize=13, color="white", labelpad=10)
    ax.tick_params(axis="both", colors="white", labelsize=12)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#444")
    ax.spines["bottom"].set_color("#444")
    ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))

    plt.tight_layout()

    # Save to a temporary file and return the path
    chart_path = os.path.join(tempfile.gettempdir(), "risk_distribution.png")
    fig.savefig(chart_path, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)

    return chart_path


def analyze_transactions(file):
    """
    Main analysis pipeline invoked by the Gradio Analyze button.

    Workflow:
        1. Read and validate the CSV file
        2. Apply fraud scoring rules to each transaction
        3. Classify risk levels
        4. Compute summary statistics
        5. Generate the risk distribution chart

    Parameters
    ----------
    file : file-like
        The uploaded CSV file from Gradio.

    Returns
    -------
    tuple
        (summary_text, results_dataframe, chart_path)
    """
    # ----------------------------------------------------------
    # Step 1: Validate the upload
    # ----------------------------------------------------------
    if file is None:
        return (
            "⚠️ No file uploaded. Please upload a CSV file.",
            pd.DataFrame(),
            None,
        )

    # ----------------------------------------------------------
    # Step 2: Read the CSV
    # ----------------------------------------------------------
    try:
        df = pd.read_csv(file)
    except Exception as e:
        return (
            f"❌ Error reading CSV: {e}",
            pd.DataFrame(),
            None,
        )

    # ----------------------------------------------------------
    # Step 3: Validate required columns
    # ----------------------------------------------------------
    missing = [col for col in REQUIRED_COLUMNS if col not in df.columns]
    if missing:
        return (
            f"❌ Missing required columns: {', '.join(missing)}\n"
            f"Expected: {', '.join(REQUIRED_COLUMNS)}\n"
            f"Found: {', '.join(df.columns.tolist())}",
            pd.DataFrame(),
            None,
        )

    # ----------------------------------------------------------
    # Step 4: Coerce amount to numeric and strip country whitespace
    # ----------------------------------------------------------
    df["amount"] = pd.to_numeric(df["amount"], errors="coerce").fillna(0)
    df["country"] = df["country"].astype(str).str.strip()

    # ----------------------------------------------------------
    # Step 5: Apply fraud scoring rules
    # ----------------------------------------------------------
    df["risk_score"] = df.apply(compute_risk_score, axis=1)

    # ----------------------------------------------------------
    # Step 6: Classify risk levels
    # ----------------------------------------------------------
    df["risk_level"] = df["risk_score"].apply(classify_risk_level)

    # ----------------------------------------------------------
    # Step 7: Compute summary metrics
    # ----------------------------------------------------------
    total        = len(df)
    high_count   = int((df["risk_level"] == "HIGH").sum())
    medium_count = int((df["risk_level"] == "MEDIUM").sum())
    low_count    = int((df["risk_level"] == "LOW").sum())

    summary = (
        f"📊  SUMMARY METRICS\n"
        f"{'═' * 40}\n"
        f"  Total Transactions : {total}\n"
        f"  🔴 High Risk       : {high_count}\n"
        f"  🟡 Medium Risk     : {medium_count}\n"
        f"  🟢 Low Risk        : {low_count}\n"
        f"{'═' * 40}"
    )

    # ----------------------------------------------------------
    # Step 8: Build the results table (selected columns only)
    # ----------------------------------------------------------
    results_df = df[
        ["transaction_id", "account", "amount", "country", "risk_score", "risk_level"]
    ].copy()

    # ----------------------------------------------------------
    # Step 9: Generate the matplotlib risk distribution chart
    # ----------------------------------------------------------
    chart_path = generate_risk_chart(df)

    return summary, results_df, chart_path


print("✅ Analysis function defined.")

## Section 5: Gradio Interface

Build the interactive UI:
- **Upload** a CSV file
- **Click Analyze** to run the fraud scoring pipeline
- **View** the summary, scored results table, and risk distribution chart

In [ ]:
# ============================================================
# Section 5: Gradio Interface
# Defines the complete UI layout and wires the Analyze button
# to the analysis pipeline.
# ============================================================

# Define a polished dark theme for the interface
theme = gr.themes.Base(
    primary_hue="blue",
    neutral_hue="slate",
    font=gr.themes.GoogleFont("Inter"),
)

# Build the Gradio Blocks layout
demo = gr.Blocks(theme=theme, title="Financial Crime Intelligence")

with demo:
    # ---- Header ----
    gr.Markdown(
        """
        # 🛡️ Financial Crime Intelligence
        ### Rule-Based Fraud Detection Engine
        Upload a CSV of transactions, click **Analyze**, and review risk scores.
        """
    )

    with gr.Row():
        # ---- Left column: Upload & Controls ----
        with gr.Column(scale=1):
            # CSV upload component
            csv_upload = gr.File(
                label="📁 Upload Transaction CSV",
                file_types=[".csv"],
                type="filepath",
            )

            # Analyze button
            analyze_btn = gr.Button(
                "🔍 Analyze Transactions",
                variant="primary",
                size="lg",
            )

            # Expected CSV format hint
            gr.Markdown(
                """
                **Expected CSV columns:**
                ```
                transaction_id, account, amount, country, timestamp
                ```

                **Scoring Rules:**
                - Amount > 500K → +50 pts
                - Amount > 1M → +20 additional pts
                - High-risk country → +30 pts
                - Max score: 100
                """
            )

        # ---- Right column: Summary ----
        with gr.Column(scale=2):
            # Summary statistics text box
            summary_output = gr.Textbox(
                label="📊 Summary Statistics",
                lines=8,
                interactive=False,
            )

    # ---- Results Table (full width) ----
    gr.Markdown("### 📋 Scored Transactions")
    results_table = gr.Dataframe(
        label="Results",
        headers=["transaction_id", "account", "amount", "country", "risk_score", "risk_level"],
        interactive=False,
    )

    # ---- Risk Distribution Chart ----
    gr.Markdown("### 📈 Risk Distribution Chart")
    chart_output = gr.Image(
        label="Risk Level Distribution",
        type="filepath",
    )

    # ---- Wire the Analyze button to the analysis function ----
    analyze_btn.click(
        fn=analyze_transactions,
        inputs=[csv_upload],
        outputs=[summary_output, results_table, chart_output],
    )

print("✅ Gradio interface built.")

## Section 6: Launch Application

Start the Gradio server. The UI will appear inline in this notebook.

> **Tip:** Set `share=True` if you want a public URL to share with others.

In [ ]:
# ============================================================
# Section 6: Launch Application
# Starts the Gradio server and renders the UI inline.
# - inline=True  → display inside the notebook
# - share=False  → local access only (set True for public URL)
# ============================================================

demo.launch(inline=True, share=False)

# When finished, you can stop the server by interrupting the kernel.